# Занятие 03. Дерево решений по внешнему набору данных: Processed Data for EV Powertrain Efficiency, Mendeley Data

## Теоретический блок

Классификация (classification) - задача отнесения наблюдения к одному из
заранее заданных классов. В данном блокноте целевая переменная
`is_allowed` кодирует условно допустимый режим: КПД электропривода не ниже учебного порога.
Дерево решений (Decision Tree) используется как интерпретируемая модель,
поскольку его правила можно записать в виде пороговых инженерных условий.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]

PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "data").exists() and (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Не найден корень проекта appai_lab.")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_FILE = DATA_DIR / "external/mendeley_ev_powertrain_efficiency_features.csv"
DIAGNOSTICS_FILE = DATA_DIR / "external/mendeley_ev_powertrain_efficiency_diagnostics.csv"
METADATA_FILE = DATA_DIR / "external/mendeley_ev_powertrain_efficiency_metadata.md"
FALLBACK_FEATURES_FILE = DATA_DIR / "practice_02_motor_efficiency_features.csv"
FALLBACK_DIAGNOSTICS_FILE = DATA_DIR / "practice_02_motor_efficiency_diagnostics.csv"

RANDOM_STATE = 20260507
GROUP_COLUMN = "profile_id"
TIME_COLUMN = "time_index"
DATASET_ID = "mendeley_ev_powertrain_efficiency"
DATASET_TITLE = "Processed Data for EV Powertrain Efficiency, Mendeley Data"

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11


def load_external_or_fallback():
    if FEATURES_FILE.exists() and DIAGNOSTICS_FILE.exists():
        features = pd.read_csv(FEATURES_FILE)
        diagnostics = pd.read_csv(DIAGNOSTICS_FILE)
        source_status = "external"
    else:
        features = pd.read_csv(FALLBACK_FEATURES_FILE)
        diagnostics = pd.read_csv(FALLBACK_DIAGNOSTICS_FILE)
        source_status = "fallback_base"
    full = features.merge(diagnostics, on="sample_id", how="left", validate="one_to_one")
    return features, diagnostics, full, source_status


def plot_correlation_heatmap(data, columns, title):
    corr = data[columns].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(7, 0.75 * len(columns)), max(5, 0.65 * len(columns))))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_xticks(range(len(columns)))
    ax.set_yticks(range(len(columns)))
    ax.set_xticklabels(columns, rotation=45, ha="right")
    ax.set_yticklabels(columns)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="Коэффициент корреляции")
    for i in range(len(columns)):
        for j in range(len(columns)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()


def group_holdout_split(data, group_column, test_share=0.25):
    groups = np.array(sorted(data[group_column].dropna().unique()))
    test_count = max(1, int(np.ceil(len(groups) * test_share)))
    test_groups = groups[-test_count:]
    test_mask = data[group_column].isin(test_groups)
    return data.index[~test_mask], data.index[test_mask], test_groups


features_df, diagnostics_df, full_df, source_status = load_external_or_fallback()
print("Источник данных:", source_status)
print("Файл признаков:", FEATURES_FILE if source_status == "external" else FALLBACK_FEATURES_FILE)
print("Размер feature-таблицы:", features_df.shape)
features_df.head()


In [ ]:
# TODO: заполните признаки строгой классификационной модели.
classification_features = []
classification_target = 'is_allowed'
if not classification_features:
    raise ValueError('Заполните classification_features.')
forbidden_classification_features = set(['drivetrain_efficiency', 'motor_efficiency', 'Powertrain_efficiency_gear_SG', 'efficiency_limit', 'class_label', 'is_allowed'])
leaked_classification_features = forbidden_classification_features & set(classification_features)
if leaked_classification_features:
    raise AssertionError(
        "Обнаружена утечка данных в classification_features: "
        f"{sorted(leaked_classification_features)}"
    )

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree

model_df = full_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=classification_features + [classification_target]
).copy()
train_idx, test_idx, test_groups = group_holdout_split(model_df, GROUP_COLUMN, test_share=0.25)
X_train = model_df.loc[train_idx, classification_features]
X_test = model_df.loc[test_idx, classification_features]
y_train = model_df.loc[train_idx, classification_target].astype(int)
y_test = model_df.loc[test_idx, classification_target].astype(int)

print("Тестовые группы:", test_groups)
print("Распределение классов в полной выборке:")
print(model_df[classification_target].value_counts().sort_index())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
class_counts = model_df[classification_target].value_counts().sort_index()
ax.bar(class_counts.index.astype(str), class_counts.values, color=["#e45756", "#54a24b"][:len(class_counts)])
ax.set_xlabel("Класс")
ax.set_ylabel("Число наблюдений")
ax.set_title("Распределение классов")
plt.tight_layout()
plt.show()


In [ ]:
def classification_metrics(y_true, y_pred):
    cm_local = confusion_matrix(y_true, y_pred, labels=[0, 1])
    not_allowed_total = cm_local[0].sum()
    dangerous_false_allowed_rate = cm_local[0, 1] / not_allowed_total if not_allowed_total > 0 else 0.0
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_not_allowed": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "recall_not_allowed": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1_not_allowed": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "dangerous_false_allowed_rate": dangerous_false_allowed_rate,
    }

majority_class = int(y_train.mode().iloc[0])
baseline_pred = np.full(len(y_test), majority_class, dtype=int)

tree_model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)
y_pred = tree_model.predict(X_test)

metrics_df = pd.DataFrame(
    [
        {"model": "majority_baseline", **classification_metrics(y_test, baseline_pred)},
        {"model": "strict_decision_tree", **classification_metrics(y_test, y_pred)},
    ]
).set_index("model")
metrics_df


In [ ]:
depth_rows = []
for depth in range(1, 9):
    depth_model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=8, random_state=RANDOM_STATE)
    depth_model.fit(X_train, y_train)
    depth_pred = depth_model.predict(X_test)
    depth_rows.append({"max_depth": depth, **classification_metrics(y_test, depth_pred)})

depth_metrics_df = pd.DataFrame(depth_rows).set_index("max_depth")

fig, ax = plt.subplots(figsize=(8, 4))
for metric in ["accuracy", "recall_not_allowed", "f1_not_allowed"]:
    ax.plot(depth_metrics_df.index, depth_metrics_df[metric], marker="o", label=metric)
ax.set_xlabel("Максимальная глубина дерева")
ax.set_ylabel("Значение метрики")
ax.set_title("Влияние глубины дерева на качество классификации")
ax.set_ylim(0.0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

depth_metrics_df


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["недопустимый", "допустимый"])
disp.plot(values_format="d", cmap="Blues")
plt.title("Матрица ошибок строгого дерева решений")
plt.show()


In [ ]:
label_candidates = ["mode_label", "class_label", "fault_label", "fault_code"]
available_label_columns = [column for column in label_candidates if column in diagnostics_df.columns]

test_error_details = model_df.loc[test_idx, ["sample_id", classification_target]].copy()
test_error_details["true_class"] = y_test
test_error_details["predicted_class"] = y_pred
test_error_details["error_type"] = np.select(
    [
        (test_error_details["true_class"] == 0) & (test_error_details["predicted_class"] == 1),
        (test_error_details["true_class"] == 1) & (test_error_details["predicted_class"] == 0),
        test_error_details["true_class"] == test_error_details["predicted_class"],
    ],
    ["false_allowed", "false_blocked", "correct"],
    default="other",
)

if available_label_columns:
    test_error_details = test_error_details.merge(
        diagnostics_df[["sample_id"] + available_label_columns],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    summary_column = available_label_columns[0]
    error_summary = (
        test_error_details.groupby([summary_column, "error_type"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )
else:
    error_summary = test_error_details["error_type"].value_counts().to_frame("count")

error_summary


In [ ]:
importance = pd.Series(tree_model.feature_importances_, index=classification_features).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
importance.plot(kind="barh", ax=ax, color="#4c78a8")
ax.set_xlabel("Относительная важность")
ax.set_title("Важность признаков строгого дерева решений")
plt.tight_layout()
plt.show()
importance.to_frame("importance")


## Демонстрация риска утечки данных

Если в классификацию включить столбцы, из которых непосредственно построена
целевая переменная, дерево будет воспроизводить правило разметки, а не решать
независимую прогностическую задачу.


In [ ]:
leakage_features = [column for column in ['vehicle_speed_m_s', 'acceleration_m_s2', 'slope_rad', 'motor_speed_rpm', 'motor_torque_nm', 'motor_efficiency', 'drivetrain_efficiency'] if column in full_df.columns]
leakage_df = full_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=leakage_features + [classification_target]
).copy()
if len(leakage_features) >= 2 and leakage_df[classification_target].nunique() == 2:
    train_idx_l, test_idx_l, _ = group_holdout_split(leakage_df, GROUP_COLUMN, test_share=0.25)
    leakage_tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=RANDOM_STATE)
    leakage_tree.fit(leakage_df.loc[train_idx_l, leakage_features], leakage_df.loc[train_idx_l, classification_target].astype(int))
    leakage_pred = leakage_tree.predict(leakage_df.loc[test_idx_l, leakage_features])
    leakage_metrics = pd.Series(
        classification_metrics(leakage_df.loc[test_idx_l, classification_target].astype(int), leakage_pred),
        name="leakage_demo",
    )
else:
    leakage_metrics = pd.Series(dtype=float, name="leakage_demo")
leakage_metrics.to_frame("value")


In [ ]:
experiment_depth = None
if experiment_depth is None:
    raise ValueError('Задайте experiment_depth.')
experiment_tree = DecisionTreeClassifier(max_depth=experiment_depth, min_samples_leaf=8, random_state=RANDOM_STATE)
experiment_tree.fit(X_train, y_train)
experiment_pred = experiment_tree.predict(X_test)
pd.Series(classification_metrics(y_test, experiment_pred), name="experiment")


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_model,
    feature_names=classification_features,
    class_names=["недопустимый", "допустимый"],
    filled=True,
    rounded=True,
    impurity=True,
)
plt.title("Строгое дерево решений")
plt.show()


## Задание

1. Укажите целевую переменную, кодировку классов и физический смысл класса 0.
2. Сравните дерево решений с базовой моделью большинства.
3. Постройте матрицу ошибок и рассчитайте долю ложных разрешений.
4. Выпишите 2-3 правила дерева в инженерной форме.
5. Объясните, какие признаки были исключены из-за риска утечки данных.
